# Titanic: Machine Learning from Disaster

The **Titanic Kaggle Competition** is one of the most popular beginner-friendly challenges in data science. It provides a real-world historical dataset and invites participants to apply data analysis and machine learning techniques to a classic predictive modeling problem.

## Background

On **April 15, 1912**, during her maiden voyage, the RMS Titanic sank after colliding with an iceberg. Of the 2,224 passengers and crew on board, more than 1,500 lost their lives, making it one of the deadliest commercial peacetime maritime disasters in modern history.

A striking aspect of this tragedy was that survival chances varied significantly depending on factors such as **gender, age, social class, and ticket fare**. For example, women, children, and higher-class passengers had higher survival rates.

## Objective

The goal of this competition is to **predict whether a passenger survived or not**, based on the information provided in the dataset. This is framed as a **binary classification problem**:

- **Target variable**: `Survived` (1 = Survived, 0 = Did not survive)  
- **Features**: Passenger characteristics such as `Age`, `Sex`, `Pclass`, `Fare`, `Embarked`, etc.

## Why This Case?

The Titanic dataset is a classic starting point because it is:
- **Small and structured**: manageable for beginners while still requiring data cleaning and preprocessing.  
- **Historically meaningful**: rooted in a real event that adds context to the data.  
- **Open-ended**: allows experimenting with a wide range of models, from logistic regression to advanced ensemble methods.

## Key Steps in the Analysis

1. **Exploratory Data Analysis (EDA):** Understand the dataset, distributions, and correlations.  
2. **Data Cleaning & Feature Engineering:** Handle missing values, encode categorical variables, and create meaningful features.  
3. **Modeling:** Train machine learning models (e.g., Logistic Regression, Decision Trees, Random Forest, Gradient Boosting).  
4. **Evaluation:** Measure performance using accuracy or other metrics, comparing predictions to actual survival outcomes.  
5. **Submission:** Generate predictions on the test set and submit results to Kaggle for scoring.

---

This competition is a hands-on introduction to predictive modeling and a great way to build practical skills in **data preprocessing, feature engineering, and machine learning pipelines**.


In [1]:
import pandas as pd
import numpy as np

import sys
sys.path.append('../..')
from sand import eda

train_df = pd.read_csv('data/train.csv')

In [2]:
# Define the specific dates range
start_date = "2023-06-01"
end_date = "2023-09-30"

safra = []
# Generate a random date within the specified range
for i in range(train_df.shape[0]):
    safra.append(
        pd.to_datetime(
            np.random.choice(pd.date_range(start=start_date, end=end_date))
        ).strftime("%Y%m")
    )

train_df["safra"] = safra

In [3]:
train_df = train_df.replace({'NaN': np.nan, 'None': np.nan, 'null': np.nan, '': np.nan})

In [4]:
sandeda = eda.SandEDA(train_df, "Survived", "safra", "PassengerId")
sandeda.report('Titanic_0_0_1')

In [6]:

# Focus a simple model using the most relevant features

# Based on IV and MI, we can focus on the following features:
# Pclass, Age, Fare, Sex, SibSp, Parch, Embarked

train_df = train_df[['Survived', 'Cabin', 'Pclass', 'Age', 'Fare', 'Sex', 'SibSp', 'Parch', 'Embarked']]

# Handle missing values
train_df['Age'] = train_df['Age'].fillna(train_df['Age'].median())
train_df['Cabin'] = train_df['Cabin'].fillna('Unknown')
train_df['Embarked'] = train_df['Embarked'].fillna(train_df['Embarked'].mode()[0])

# Encode categorical variables
train_df['Sex'] = train_df['Sex'].apply(lambda x: 1 if x == 'male' else 0)
train_df['Embarked'] = train_df['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})
train_df['Cabin'] = train_df['Cabin'].apply(lambda x: 0 if x == 'Unknown' else 1)


In [7]:
from sand import logisticregression as lr

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    train_df.values[:, 1:], train_df.values[:, 0], test_size=0.2, random_state=42
)


intercept, coef = lr.fit(X_train, 
                        y_train,
                        lr=0.001, 
                        epochs=2000,
                        batch_size=60,
                        l2_lambda=None,
                        decay=None)

Epoch 0, Log Loss: 0.6544
Epoch 500, Log Loss: 0.5546
Epoch 1000, Log Loss: 0.5241
Epoch 1500, Log Loss: 0.5058


In [9]:
print("Intercept:", round(intercept, 3))
feature_names = train_df.columns[1:]
for name, c in zip(feature_names, coef):
    print(f"{name}: {round(c, 3)}")

Intercept: 0.321
Cabin: 0.474
Pclass: -0.13
Age: -0.005
Fare: 0.008
Sex: -1.451
SibSp: -0.212
Parch: 0.064
Embarked: 0.258


In [10]:
from sklearn.metrics import roc_auc_score, accuracy_score

X = X_test
y_true = y_test

y_pred_proba = lr.predict_proba(X, intercept, coef)
y_pred = [1 if p >= 0.5 else 0 for p in y_pred_proba]

roc_auc = roc_auc_score(y_true, y_pred_proba)
accuracy = accuracy_score(y_true, y_pred)

print(f"ROC AUC: {roc_auc:.4f}")
print(f"Accuracy: {accuracy:.4f}")

ROC AUC: 0.8687
Accuracy: 0.7933


In [11]:
intercept, coef = lr.fit_newton(X_train, y_train, reg_lambda=1.0, tol=1e-6, max_iter=100)

In [12]:
print("Intercept:", intercept)
feature_names = train_df.columns[1:]
for name, c in zip(feature_names, coef):
    print(f"{name}: {c}")

Intercept: 3.652947862424013
Cabin: [ 6.06087282e-01 -7.60596234e-01 -3.15593156e-02  2.25497579e-03
 -2.59131830e+00 -3.01708979e-01 -1.10677772e-01  2.10070097e-01]


In [13]:
from sklearn.metrics import roc_auc_score, accuracy_score

X = X_test
y_true = y_test

y_pred_proba = intercept + X_test @ coef.T
y_pred = [1 if p >= 0.5 else 0 for p in y_pred_proba]

roc_auc = roc_auc_score(y_true, y_pred_proba)
accuracy = accuracy_score(y_true, y_pred)

print(f"ROC AUC: {roc_auc:.4f}")
print(f"Accuracy: {accuracy:.4f}")

ROC AUC: 0.8797
Accuracy: 0.8101


In [14]:
from sklearn.linear_model import LogisticRegression

# Fit logistic regression using sklearn
model = LogisticRegression(max_iter=2000)
model.fit(X_train, y_train)

# Predict probabilities and classes
y_pred_proba_sklearn = model.predict_proba(X_test)[:, 1]
y_pred_sklearn = model.predict(X_test)

# Evaluate
roc_auc_sklearn = roc_auc_score(y_test, y_pred_proba_sklearn)
accuracy_sklearn = accuracy_score(y_test, y_pred_sklearn)

print(f"sklearn ROC AUC: {roc_auc_sklearn:.4f}")
print(f"sklearn Accuracy: {accuracy_sklearn:.4f}")

sklearn ROC AUC: 0.8797
sklearn Accuracy: 0.8101


In [23]:
# Read the test data
test_df = pd.read_csv('data/test.csv')

In [24]:
# Define the specific dates range
start_date = "2023-06-01"
end_date = "2023-09-30"

safra = []
# Generate a random date within the specified range
for i in range(test_df.shape[0]):
    safra.append(
        pd.to_datetime(
            np.random.choice(pd.date_range(start=start_date, end=end_date))
        ).strftime("%Y%m")
    )

test_df["safra"] = safra

In [25]:
test_df = test_df.replace({'NaN': np.nan, 'None': np.nan, 'null': np.nan, '': np.nan})

In [33]:
np.random.seed(42)
test_df['Survived'] = np.random.choice([0, 1], size=test_df.shape[0], p=[0.6, 0.4])

In [34]:
sandeda = eda.SandEDA(test_df, "Survived", "safra", "PassengerId")
sandeda.report('Titanic_0_0_2')

In [40]:
# Read the test data
test_df = pd.read_csv('data/test.csv')

# Preprocess test data using the same steps as train_df
test_df['Age'] = test_df['Age'].fillna(train_df['Age'].median())
test_df['Cabin'] = test_df['Cabin'].fillna('Unknown')
test_df['Embarked'] = test_df['Embarked'].fillna(train_df['Embarked'].mode()[0])
test_df['Sex'] = test_df['Sex'].apply(lambda x: 1 if x == 'male' else 0)
test_df['Embarked'] = test_df['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})
test_df['Cabin'] = test_df['Cabin'].apply(lambda x: 0 if x == 'Unknown' else 1)
test_df['Fare'] = test_df['Fare'].fillna(train_df['Fare'].median())

# Select the same features as used for training
X_submit = test_df[['Cabin', 'Pclass', 'Age', 'Fare', 'Sex', 'SibSp', 'Parch', 'Embarked']].values

# Predict using sklearn model
y_submit_pred = model.predict(X_submit)

# Prepare submission DataFrame
submission = pd.DataFrame({
    'PassengerId': test_df['PassengerId'],
    'Survived': y_submit_pred.astype(int)
})

submission.to_csv('submission.csv', index=False)